# STEP-046 — Robustness evaluation of MODEL0001 (thin caller)

All logic lives in `17_Automation/robustness`; every perturbation and level lives in `08_Evaluation/EVAL0001/robustness/config.yaml`. Do not add code or parameters here. Runbook: `08_Evaluation/EVAL0001/robustness/KAGGLE_RUNBOOK.md`.

**Notebook settings:** Accelerator `GPU T4 x2`, Internet `On`, inputs `isthisdeception/ds0001-pp0001` and `isthisdeception/thesis-experiment-artifacts`, secret `GITHUB_TOKEN`.

Run every cell in order: 0 Setup → A Inference → B Analysis → C Push. If the session is interrupted, run 0 and A again: finished conditions are reused.

## 0. Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/isthisdeception/thesis.git'
BRANCH = 'main'
REPO = Path('/kaggle/working/thesis')
PLAN = 'kaggle_sync/sync_plan.robustness.yaml'
AUTOMATION = REPO / '17_Automation'
OUT = REPO / '08_Evaluation' / 'EVAL0001' / 'robustness'

if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
print(subprocess.run(['git', '-C', str(REPO), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# GITHUB_TOKEN (Add-ons -> Secrets) is needed only for C.
try:
    from kaggle_secrets import UserSecretsClient
    try:
        os.environ['GITHUB_TOKEN'] = UserSecretsClient().get_secret('GITHUB_TOKEN')
        print('GITHUB_TOKEN loaded')
    except Exception:
        print('GITHUB_TOKEN not set: C will fail')
except ImportError:
    print('kaggle_secrets unavailable (not on Kaggle)')

In [ ]:
# Pull latest main, install environment/kaggle-requirements.txt, verify both inputs are attached.
!cd {AUTOMATION} && python -m kaggle_sync --plan {PLAN} bootstrap

In [ ]:
import torch, fastai
from importlib.metadata import version
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())
print('fastai', fastai.__version__, '| plum-dispatch', version('plum-dispatch'))
print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'GPU not visible: set Accelerator to GPU T4 x2'
!cd {AUTOMATION} && python -m robustness conditions | tail -n 3

## A. Inference (about 30–60 min)

Finds `MODEL0001_export.pkl` in the artifact input and checks its SHA-256. Runs the clean condition first: it must print `clean check PASS`. If it stops with `CleanCheckError`, do not continue; report the message. Then one line per condition. Re-running this cell skips finished conditions.

In [ ]:
!cd {AUTOMATION} && python -m robustness run

## B. Analysis (under a minute)

In [ ]:
!cd {AUTOMATION} && python -m robustness analyze
import pandas as pd
pd.set_option('display.width', 200)
print(pd.read_csv(OUT / 'robustness_summary.csv')[['perturbation', 'tolerance_level', 'worst_level', 'delta_balanced_accuracy', 'delta_roc_auc', 'pattern']].to_string(index=False))

## C. Push results to GitHub (needs GITHUB_TOKEN)

Pushes only `08_Evaluation/EVAL0001/robustness` to a `kaggle/` branch; it reaches `main` through a reviewed PR.

In [ ]:
from datetime import datetime
PUSH_BRANCH = f"kaggle/robustness-{datetime.now():%Y%m%d-%H%M}"
!cd {AUTOMATION} && python -m kaggle_sync --plan {PLAN} push-metadata --branch {PUSH_BRANCH} --commit-message "sync: EVAL0001 robustness results from Kaggle"
print('pushed to', PUSH_BRANCH)